# IEEE-CIS E-Commerce Fraud Detection: Causal E10 Ensemble Demo

This notebook automatically bootstraps the project repository from GitHub, initializes Git LFS to download the models, installs dependencies, and launches the live inference web application for the strictly causal E10 frozen CatBoost ensemble (Test F1: 62.25%).

In [ ]:
import os
import sys
import subprocess
import shutil

# ==========================================
# CONFIGURATION
# ==========================================
REPO_URL = "https://github.com/sanjais146/anomoly_detection"

def run_cmd(cmd):
    print(f"> {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error:\n{result.stderr}")
    return result.returncode == 0

def bootstrap():
    print("Step 1: Bootstrapping Repository from GitHub...")
    repo_name = REPO_URL.rstrip('/').split('/')[-1]
    if repo_name.endswith('.git'):
        repo_name = repo_name[:-4]

    # Clone if not already present, or clean up broken clones
    if os.path.exists(repo_name):
        print(f"Found existing directory '{repo_name}'. Checking integrity...")
        if not os.path.exists(f"{repo_name}/app") or not os.path.exists(f"{repo_name}/models"):
            print("Directory appears corrupted or incomplete. Removing...")
            shutil.rmtree(repo_name, ignore_errors=True)

    if not os.path.exists(repo_name) and not os.path.exists('app') and not os.path.exists('models'):
        print(f"Cloning {REPO_URL}...")
        run_cmd(f"git clone {REPO_URL}")

    # Resolve project root dynamically
    repo_root = None
    for root, dirs, files in os.walk('/content'):
        if 'app' in dirs and 'models' in dirs and 'frontend' in dirs:
            repo_root = root
            break

    # Fallback for local Jupyter execution outside Colab
    if not repo_root and os.path.exists('app') and os.path.exists('models'):
        repo_root = os.getcwd()

    if repo_root:
        print(f"\u2705 Found project root at: {repo_root}")
        os.chdir(repo_root)
        sys.path.insert(0, repo_root)
        
        print("Initializing Git LFS and pulling models...")
        run_cmd("git lfs install")
        run_cmd("git lfs pull")
    else:
        print("\u274c ERROR: Could not resolve project root. The repository may not have cloned correctly.")
        return False
    return True

if not bootstrap():
    raise RuntimeError("Bootstrapping failed.")

In [ ]:
def verify_files():
    print("Step 2: Verifying required files and models...")
    required_dirs = ['app', 'models', 'frontend']
    required_files = [
        'models/e10_base.cbm',
        'models/e10_deep.cbm',
        'models/e10_weight.cbm',
        'app/main.py',
        'app/predictor.py',
        'colab/run_demo.ipynb'
    ]
    
    missing = False
    for d in required_dirs:
        if not os.path.isdir(d):
            print(f"\u274c Missing directory: {d}/")
            missing = True
            
    for f in required_files:
        if not os.path.isfile(f):
            print(f"\u274c Missing file: {f}")
            missing = True
        else:
            # Verify LFS pulled correctly by checking file size (should be > 1MB)
            size_mb = os.path.getsize(f) / (1024 * 1024)
            if f.endswith('.cbm') and size_mb < 1:
                print(f"\u274c ERROR: Model file {f} is too small ({size_mb:.2f} MB). Git LFS pull failed!")
                missing = True
    
    if missing:
        print("\u274c ERROR: Required files are missing or Git LFS failed to pull the actual model binaries.")
        return False
    else:
        print("\u2705 All required directories and E10 model artifacts verified.")
        return True

if not verify_files():
    raise RuntimeError("Verification failed.")

In [ ]:
print("Step 3: Installing dependencies...")
!pip install -q fastapi uvicorn catboost pandas numpy pydantic requests pyngrok
print("\u2705 Dependencies installed.")

In [ ]:
import threading
import time
import requests
import uvicorn
import json
from app.main import app
from pyngrok import ngrok

def start_deployment():
    print("Step 4: Authenticating ngrok...")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('NGROK_AUTHTOKEN')
    except Exception:
        token = os.environ.get('NGROK_AUTHTOKEN')
    
    if not token:
        print("\u274c ERROR: NGROK_AUTHTOKEN not found.")
        print("To fix this:")
        print("1. Go to Colab sidebar (key icon) -> Secrets")
        print("2. Add a new secret with Name: NGROK_AUTHTOKEN")
        print("3. Value: <your_ngrok_token>")
        print("4. Enable 'Notebook access'")
        print("\nMake sure you have added the token, then rerun this cell.")
        return False
    
    print("Step 5: Starting FastAPI (E10 Ensemble)...")
    def run_api():
        uvicorn.run(app, host="127.0.0.1", port=8000, log_level="critical")
    
    api_thread = threading.Thread(target=run_api, daemon=True)
    api_thread.start()
    
    print("Step 6: Polling health check...")
    for i in range(15):
        try:
            r = requests.get("http://127.0.0.1:8000/health")
            if r.status_code == 200:
                print("\u2705 FastAPI is ONLINE and E10 models are LOADED.")
                break
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(1)
    else:
        print("\u274c ERROR: FastAPI failed to start or health check timed out.")
        return False
    
    print("Step 7: Creating ngrok tunnel...")
    ngrok.set_auth_token(token)
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url
    
    print("\n" + "="*40)
    print("E10 E-COMMERCE FRAUD DETECTION DEMO")
    print("="*40)
    print("FastAPI: ONLINE")
    print("E10 Model: LOADED")
    print("ngrok: ONLINE\n")
    print("Public Dashboard:")
    print(f"{public_url}\n")
    print("API Documentation:")
    print(f"{public_url}/docs")
    print("="*40 + "\n")
    
    print("Step 8: Running synthetic smoke test...")
    sample_tx = {
        "TransactionAmt": 999.99,
        "ProductCD": "W",
        "card1": 10409,
        "P_emaildomain": "anonymous.com"
    }
    
    try:
        response = requests.post("http://127.0.0.1:8000/predict", json=sample_tx)
        if response.status_code == 200:
            print("\u2705 Synthetic Prediction Successful!\n")
            print(json.dumps(response.json(), indent=2))
        else:
            print(f"\u274c API returned status code: {response.status_code}")
            print(response.text)
    except Exception as e:
        print(f"\u274c Smoke test failed: {e}")

start_deployment()